# 🧠 Baseline Autoencoder for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements a **simple fully-connected autoencoder** as a baseline model for anomaly detection in brain MRI scans.

### Model Architecture
- **Type**: Fully-Connected Autoencoder
- **Input**: 128×128 grayscale MRI slice (flattened to 16,384 pixels)
- **Latent Dimension**: 128
- **Total Parameters**: ~8.5M

### Training Strategy
- **Training Data**: IXI dataset (normal healthy brains)
- **Test Data**: BraTS 2021 (brains with tumors)
- **Loss Function**: Combined MSE + MS-SSIM (α=0.84)
- **Optimizer**: Adam with weight decay

### Expected Performance
- **AUROC**: ~0.75-0.80 (baseline)
- **Training Time**: ~3-4 hours on Colab GPU (20 epochs, batch_size=64)

---

## 1️⃣ Setup and Environment Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
# Keep Colab session alive (prevents random disconnects)
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button"); 
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click connect button");
     btn.click();
   }
 }
 setInterval(ClickConnect, 60000)
'''))

print("✅ Keep-alive script activated!")

In [ ]:
# Install required packages
!pip install pytorch-msssim -q

print("✅ All packages installed!")

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pytorch_msssim import SSIM  # Changed from MS_SSIM to SSIM

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from glob import glob
import os
import time
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

print("✅ All libraries imported successfully!")

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"
IXI_TRAIN_PATH = f"{BASE_PATH}/data/processed_ixi/train"
IXI_VAL_PATH = f"{BASE_PATH}/data/processed_ixi/val"
BRATS_PATH = f"{BASE_PATH}/data/brats2021_test"
MODEL_PATH = f"{BASE_PATH}/models/saved_models"
RESULTS_PATH = f"{BASE_PATH}/results/ae_baseline"

# Create directories if they don't exist
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("📁 Paths configured:")
print(f"   IXI Train: {IXI_TRAIN_PATH}")
print(f"   IXI Val: {IXI_VAL_PATH}")
print(f"   BraTS Data: {BRATS_PATH}")
print(f"   Models: {MODEL_PATH}")
print(f"   Results: {RESULTS_PATH}")

## 2️⃣ Data Loading and Preprocessing

In [ ]:
Cannot connect to GPU backend
You cannot currently connect to a GPU due to usage limits in Colab. Learn more
To get more access to GPUs, consider purchasing Colab compute units with Pay As You Go.

In [ ]:
# Load file paths
print("📂 Loading file paths...")

train_files = sorted(glob(f"{IXI_TRAIN_PATH}/*.npy"))
val_files = sorted(glob(f"{IXI_VAL_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))

print(f"✅ Found {len(train_files)} IXI training slices")
print(f"✅ Found {len(val_files)} IXI validation slices")
print(f"✅ Found {len(brats_files)} BraTS test slices (tumors)")

In [ ]:
# Data already split - just display statistics
print(f"📊 Data Split (pre-configured):")
print(f"   Training: {len(train_files)} slices")
print(f"   Validation: {len(val_files)} slices")
print(f"   Test (BraTS): {len(brats_files)} slices")

In [ ]:
# Create datasets
train_dataset = MRIDataset(train_files)
val_dataset = MRIDataset(val_files)
test_dataset = MRIDataset(brats_files)

# Create dataloaders
BATCH_SIZE = 64  # Increased from 32 for fewer iterations (2x faster per epoch)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

# Normal brains (IXI)
for i in range(8):
    img = np.load(train_files[i])
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Normal {i+1}', fontsize=10)

# Tumor brains (BraTS)
for i in range(8):
    img = np.load(brats_files[i])
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title(f'Tumor {i+1}', fontsize=10)

axes[0, 0].set_ylabel('IXI (Normal)', fontsize=12)
axes[1, 0].set_ylabel('BraTS (Tumor)', fontsize=12)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/baseline_data_samples.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Sample images visualized!")

## 3️⃣ Model Architecture

In [ ]:
class BaselineAutoencoder(nn.Module):
    """
    Simple Fully-Connected Autoencoder
    
    Architecture:
    - Input: 128×128×1 = 16,384 pixels (flattened)
    - Encoder: 16384 -> 512 -> 256 -> 128 (latent)
    - Decoder: 128 -> 256 -> 512 -> 16384
    - Output: Reshaped to 128×128×1
    
    Features:
    - Batch normalization for stable training
    - ReLU activation
    - Dropout (0.2) for regularization
    - Sigmoid output for [0,1] pixel values
    """
    
    def __init__(self, input_dim=16384, latent_dim=128):
        super(BaselineAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder: Compress to latent space
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            nn.Dropout(0.2),
            
            nn.Linear(256, latent_dim),
            nn.ReLU(True)
        )
        
        # Decoder: Reconstruct from latent space
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Dropout(0.2),
            
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # Output in [0, 1] range
        )
    
    def forward(self, x):
        """
        Forward pass
        Args:
            x: Input tensor of shape (batch_size, 1, 128, 128)
        Returns:
            Reconstructed tensor of shape (batch_size, 1, 128, 128)
        """
        batch_size = x.size(0)
        
        # Flatten input: (B, 1, 128, 128) -> (B, 16384)
        x_flat = x.view(batch_size, -1)
        
        # Encode
        z = self.encoder(x_flat)
        
        # Decode
        x_recon = self.decoder(z)
        
        # Reshape to image: (B, 16384) -> (B, 1, 128, 128)
        x_recon = x_recon.view(batch_size, 1, 128, 128)
        
        return x_recon
    
    def get_latent(self, x):
        """
        Get latent representation
        Args:
            x: Input tensor of shape (batch_size, 1, 128, 128)
        Returns:
            Latent tensor of shape (batch_size, latent_dim)
        """
        batch_size = x.size(0)
        x_flat = x.view(batch_size, -1)
        return self.encoder(x_flat)

print("✅ Baseline Autoencoder class defined!")

In [ ]:
# Create model instance
model = BaselineAutoencoder(input_dim=128*128, latent_dim=128).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("🧠 Baseline Autoencoder Created!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.2f} MB")
print(f"   Device: {next(model.parameters()).device}")

In [ ]:
# Test forward pass
sample_input = torch.randn(2, 1, 128, 128).to(device)
sample_output = model(sample_input)

print("✅ Forward pass test:")
print(f"   Input shape: {sample_input.shape}")
print(f"   Output shape: {sample_output.shape}")
print(f"   Output range: [{sample_output.min():.4f}, {sample_output.max():.4f}]")

# Test latent extraction
sample_latent = model.get_latent(sample_input)
print(f"   Latent shape: {sample_latent.shape}")

## 4️⃣ Loss Function and Optimizer

In [ ]:
class CombinedLoss(nn.Module):
    """
    Combined MSE + SSIM Loss (using regular SSIM for 128×128 images)
    
    Loss = α * MSE + (1-α) * (1 - SSIM)
    
    Args:
        alpha: Weight for MSE loss (default: 0.84)
               Higher α emphasizes pixel-wise accuracy
               Lower α emphasizes structural similarity
    """
    
    def __init__(self, alpha=0.84):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.ssim = SSIM(data_range=1.0, size_average=True, channel=1, win_size=11)
    
    def forward(self, output, target):
        mse_loss = self.mse(output, target)
        ssim_loss = 1 - self.ssim(output, target)
        
        combined = self.alpha * mse_loss + (1 - self.alpha) * ssim_loss
        return combined

# Create loss function
criterion = CombinedLoss(alpha=0.84)

print("✅ Combined Loss Function created!")
print(f"   MSE weight (α): 0.84")
print(f"   SSIM weight (1-α): 0.16")
print(f"   Note: Using SSIM (not MS-SSIM) for 128×128 images")

In [ ]:
# Create optimizer and scheduler
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

print("✅ Optimizer and Scheduler created!")
print(f"   Optimizer: Adam")
print(f"   Initial LR: 1e-3")
print(f"   Weight Decay: 1e-5")
print(f"   Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")

## 5️⃣ Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train for one epoch
    
    Returns:
        Average loss for the epoch
    """
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(dataloader, desc='Training')
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    
    return running_loss / len(dataloader)

print("✅ Training function defined!")

In [ ]:
def validate(model, dataloader, criterion, device):
    """
    Validate the model
    
    Returns:
        Average validation loss
    """
    model.eval()
    running_loss = 0.0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            loss = criterion(output, target)
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    
    return running_loss / len(dataloader)

print("✅ Validation function defined!")

In [ ]:
# Training configuration
NUM_EPOCHS = 20  # Reduced epochs (baseline converges quickly, mainly for comparison)
SAVE_INTERVAL = 5  # Save every 5 epochs (more frequent for Colab stability)

# Storage for metrics
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_epoch = 0

print("🚀 Starting Training...")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Device: {device}")
print(f"   Model: Baseline Autoencoder")
print(f"   Checkpoint Interval: Every {SAVE_INTERVAL} epochs")
print("-" * 60)

In [ ]:
# Main training loop
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Print epoch summary
    epoch_time = time.time() - epoch_start
    print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | "
          f"Train Loss: {train_loss:.6f} | "
          f"Val Loss: {val_loss:.6f} | "
          f"Time: {epoch_time:.2f}s")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, f'{MODEL_PATH}/baseline_best.pth')
        print(f"   ✅ Best model saved! (Val Loss: {val_loss:.6f})")
    
    # Save checkpoint every SAVE_INTERVAL epochs
    if (epoch + 1) % SAVE_INTERVAL == 0:
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, f'{MODEL_PATH}/baseline_epoch{epoch+1}.pth')
    
    print("-" * 60)

total_time = time.time() - start_time
print(f"\n🎉 Training Complete!")
print(f"   Total Time: {total_time/60:.2f} minutes")
print(f"   Best Epoch: {best_epoch}")
print(f"   Best Val Loss: {best_val_loss:.6f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Val Loss', linewidth=2)
plt.axvline(x=best_epoch-1, color='r', linestyle='--', label=f'Best Epoch ({best_epoch})')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Val Loss', linewidth=2)
plt.axvline(x=best_epoch-1, color='r', linestyle='--', label=f'Best Epoch ({best_epoch})')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (log scale)', fontsize=12)
plt.title('Training and Validation Loss (Log Scale)', fontsize=14, fontweight='bold')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training curves saved!")

## 6️⃣ Evaluation and Anomaly Detection

In [ ]:
# Load best model
checkpoint = torch.load(f'{MODEL_PATH}/baseline_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Best model loaded!")
print(f"   Epoch: {checkpoint['epoch']}")
print(f"   Val Loss: {checkpoint['val_loss']:.6f}")

In [ ]:
def calculate_reconstruction_error(model, dataloader, device):
    """
    Calculate reconstruction error for anomaly detection
    
    Returns:
        Array of MSE errors for each sample
    """
    model.eval()
    errors = []
    
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Computing errors'):
            data = data.to(device)
            recon = model(data)
            
            # Calculate MSE per sample
            mse = nn.functional.mse_loss(recon, data, reduction='none')
            mse = mse.view(mse.size(0), -1).mean(dim=1)
            
            errors.extend(mse.cpu().numpy())
    
    return np.array(errors)

print("✅ Error calculation function defined!")

In [ ]:
# Calculate reconstruction errors
print("📊 Calculating reconstruction errors...")

normal_errors = calculate_reconstruction_error(model, val_loader, device)
anomaly_errors = calculate_reconstruction_error(model, test_loader, device)

print(f"\n✅ Errors calculated!")
print(f"   Normal (IXI): {len(normal_errors)} samples")
print(f"   Anomaly (BraTS): {len(anomaly_errors)} samples")
print(f"\n   Normal - Mean: {normal_errors.mean():.6f}, Std: {normal_errors.std():.6f}")
print(f"   Anomaly - Mean: {anomaly_errors.mean():.6f}, Std: {anomaly_errors.std():.6f}")

In [ ]:
# Calculate AUROC
y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])

auroc = roc_auc_score(y_true, y_scores)

# Calculate AUPRC
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)

print("📈 Performance Metrics:")
print(f"   AUROC: {auroc:.4f}")
print(f"   AUPRC: {auprc:.4f}")

In [ ]:
# Plot reconstruction error distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal (IXI)', density=True, color='blue')
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly (BraTS)', density=True, color='red')
plt.xlabel('Reconstruction Error (MSE)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Reconstruction Error Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.plot(fpr, tpr, linewidth=2, label=f'Baseline AE (AUROC = {auroc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/baseline_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Evaluation plots saved!")

In [ ]:
# Visualize reconstructions
def visualize_reconstructions(model, dataloader, device, num_samples=8, title=""):
    """
    Visualize original images, reconstructions, and error maps
    """
    model.eval()
    data, _ = next(iter(dataloader))
    data = data[:num_samples].to(device)
    
    with torch.no_grad():
        recon = model(data)
    
    error = torch.abs(data - recon)
    
    fig, axes = plt.subplots(3, num_samples, figsize=(20, 6))
    
    for i in range(num_samples):
        # Original
        axes[0, i].imshow(data[i, 0].cpu(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Original', fontsize=10, fontweight='bold')
        
        # Reconstruction
        axes[1, i].imshow(recon[i, 0].cpu(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Reconstruction', fontsize=10, fontweight='bold')
        
        # Error map
        axes[2, i].imshow(error[i, 0].cpu(), cmap='hot')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_title('Error Map', fontsize=10, fontweight='bold')
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

print("✅ Visualization function defined!")

In [ ]:
# Visualize normal brain reconstructions
print("🧠 Normal Brain Reconstructions (IXI):")
fig1 = visualize_reconstructions(model, val_loader, device, num_samples=8, 
                                  title="Baseline AE - Normal Brain Reconstructions")
plt.savefig(f'{RESULTS_PATH}/baseline_normal_reconstructions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize tumor brain reconstructions
print("🔴 Tumor Brain Reconstructions (BraTS):")
fig2 = visualize_reconstructions(model, test_loader, device, num_samples=8, 
                                  title="Baseline AE - Tumor Brain Reconstructions (Anomalies)")
plt.savefig(f'{RESULTS_PATH}/baseline_tumor_reconstructions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ All visualizations saved!")

In [ ]:
# Save final results
results = {
    'model': 'Baseline Autoencoder',
    'auroc': float(auroc),
    'auprc': float(auprc),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'normal_error_mean': float(normal_errors.mean()),
    'normal_error_std': float(normal_errors.std()),
    'anomaly_error_mean': float(anomaly_errors.mean()),
    'anomaly_error_std': float(anomaly_errors.std()),
    'total_params': total_params,
    'trainable_params': trainable_params
}

# Save to file
import json
with open(f'{RESULTS_PATH}/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("💾 Results saved to JSON")
print("\n" + "="*60)
print("📊 FINAL RESULTS - BASELINE AUTOENCODER")
print("="*60)
for key, value in results.items():
    print(f"   {key}: {value}")
print("="*60)